In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


In [ ]:
import os
import requests
import time

# ======== CONFIGURATION ========
species_name = "Carpobrotus acinaciformis"  # can change species
output_folder = "inat_images"
license_filter = ["cc0", "cc-by", "cc-by-sa"]  # Licenses permitted
max_pages = 50  # Increase for more images
per_page = 100  # Max 200
sleep_time = 1  # seconds between requests

# ======== SETUP ========
os.makedirs(output_folder, exist_ok=True)
base_url = "https://api.inaturalist.org/v1/observations"

# ======== FUNCTION ========
def download_images():
    page = 1
    downloaded = 0

    while page <= max_pages:
        print(f"page {page}")
        params = {
            "taxon_name": species_name,
            "quality_grade": "research",
            "per_page": per_page,
            "page": page,
            "order": "desc",
            "order_by": "created_at",
            "photo_license": ",".join(license_filter)  # License filter
        }

        r = requests.get(base_url, params=params)
        if r.status_code != 200:
            print(f"http {r.status_code}")
            break

        data = r.json()
        results = data.get("results", [])

        if not results:
            print("no more results")
            break

        for obs in results:
            if "photos" in obs:
                for photo in obs["photos"]:
                    url = photo.get("url")
                    if url:
                        img_url = url.replace("square", "original")  # Get full resolution
                        img_id = photo.get("id")
                        img_path = os.path.join(output_folder, f"{img_id}.jpg")

                        # Skip existing
                        if os.path.exists(img_path):
                            continue

                        try:
                            img_data = requests.get(img_url).content
                            with open(img_path, "wb") as f:
                                f.write(img_data)
                            downloaded += 1
                        except Exception as e:
                            print(f"failed {img_url}: {e}")

        page += 1
        time.sleep(sleep_time)

    print(f"got {downloaded} images -> {output_folder}")

# ======== RUN ========
download_images()


Fetching page 1...
Fetching page 2...
Fetching page 3...
